#Check GPU availability

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

#LBFGS optimizer

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri Sep 11 15:50:38 2020

@author: cosmin
"""
import tensorflow as tf
import numpy as np

def tfp_function_factory(model, *args):
    """A factory to create a function required by tfp.optimizer.lbfgs_minimize.
    Based on the example from https://pychao.com/2019/11/02/optimize-tensorflow-keras-models-with-l-bfgs-from-tensorflow-probability/
    Args:
        model [in]: an instance of `tf.keras.Model` or its subclasses.
        *args: arguments to be passed to model.get_grads method

    Returns:
        A function that has a signature of:
            loss_value, gradients = f(model_parameters).
    """

    # obtain the shapes of all trainable parameters in the model
    shapes = tf.shape_n(model.trainable_variables)
    n_tensors = len(shapes)

    # we'll use tf.dynamic_stitch and tf.dynamic_partition later, so we need to
    # prepare required information first
    count = 0
    idx = [] # stitch indices
    part = [] # partition indices

    for i, shape in enumerate(shapes):
        n = np.prod(shape)
        idx.append(tf.reshape(tf.range(count, count+n, dtype=tf.int32), shape))
        part.extend([i]*n)
        count += n

    part = tf.constant(part)
    #@tf.function
    def assign_new_model_parameters(params_1d):
        """A function updating the model's parameters with a 1D tf.Tensor.

        Args:
            params_1d [in]: a 1D tf.Tensor representing the model's trainable parameters.
        """

        params_1d = tf.cast(params_1d, dtype=tf.float64)

        params = tf.dynamic_partition(params_1d, part, n_tensors)
        for i, (shape, param) in enumerate(zip(shapes, params)):
            model.trainable_variables[i].assign(tf.reshape(param, shape))

    # now create a function that will be returned by this factory
    # @tf.function
    def f(params_1d):
        """A function that can be used by tfp.optimizer.lbfgs_minimize."""
        assign_new_model_parameters(params_1d)

        # 1. Grab exactly 7 outputs from the PreBias get_grad_adam
        L, grads, FI_loss, psi_loss, n_loss, I_loss, cons_loss = model.get_grad_adam(*args)
        grads = tf.dynamic_stitch(idx, grads)

        # print out iteration & loss
        f.iter.assign_add(1)
        if f.iter % model.print_epoch == 0:
            # 2. Append to the PreBias model's tracking lists
            model.FI_bc_loss_curve.append(FI_loss.numpy())
            model.psi_loss_curve.append(psi_loss.numpy())
            model.n_loss_curve.append(n_loss.numpy())
            model.I_loss_curve.append(I_loss.numpy())
            model.cons_loss_curve.append(cons_loss.numpy())
            model.total_loss_curve.append(L.numpy())
            model.loss_phase_curve.append("lbfgs")
            model.loss_step_curve.append(int(f.iter.numpy()))

            # 3. Print exact PreBias breakdown
            print(f"L-BFGS Iter {f.iter.numpy()} | Total: {L.numpy():.4e} | FI_bc: {FI_loss.numpy():.4e} | Cons: {cons_loss.numpy():.4e} | I: {I_loss.numpy():.4e} | psi: {psi_loss.numpy():.4e} | n: {n_loss.numpy():.4e}")

        return L, grads

    # store these information as members so we can use them outside the scope
    f.iter = tf.Variable(0)
    f.idx = idx
    f.part = part
    f.shapes = shapes
    f.assign_new_model_parameters = assign_new_model_parameters

    return f

#User-defined functions

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

data_type = "float64"

# -----------------------------------------------------------
# Mesh & Grid Generation
# -----------------------------------------------------------

def xi_tanh(numpts, lamb=0.1):
    """Generates a non-uniform spatial grid clustered at boundaries."""
    i = np.arange(numpts + 1)
    s = np.arctanh(np.tanh(0.6) / (2 * lamb - 1))
    xi = 0.5 * (np.tanh(s * (2 * i / numpts - 1.0)) / np.tanh(s) + 1.0)[:, np.newaxis]
    return xi

def mesh_gen(thick, numPts, lamb=0.1):

    # Generate the grid across the thickness
    xi = thick * xi_tanh(numPts, lamb=lamb)
    return tf.convert_to_tensor(xi, dtype=data_type)

def grid2D(xspace, tspace):
    """2D grid generation for PINN collocation points."""
    X, T = np.meshgrid(xspace, tspace)
    Xgrid = np.vstack([X.flatten(), T.flatten()]).T
    return tf.cast(Xgrid, data_type)

# -----------------------------------------------------------
# Plotting
# -----------------------------------------------------------

def plot_single(x, y, x_label, y_label, title=""):
    """Plotting function"""
    plt.plot(x, y)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.grid(True)
    plt.show()

# -----------------------------------------------------------
# Utility Functions
# -----------------------------------------------------------

def dist(x, lb, ub):
    """Boundary distance function for hard-encoding boundary conditions."""
    return (x - lb) * (ub - x)

#Main PINN class

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Solve coupled Poisson, current continuity, and drift-diffusion equations
See Readme file for the problem's detail.
@author: N. Thongprong, C. Anitescu
"""
import tensorflow as tf
import numpy as np
import time
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
import scipy.constants as con
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200

class model(tf.keras.Model):

    def __init__(self, layers, train_op, num_epoch_adam, num_epoch_lbfgs, print_epoch, V_pre, Vbi, Vt, n0, nL, lambda_, I_bulk, W_cons):
        super(model, self).__init__()
        self.model_layers = layers
        self.train_op = train_op
        self.num_epoch_adam = num_epoch_adam
        self.num_epoch_lbfgs = num_epoch_lbfgs
        self.print_epoch = print_epoch
        self.loop_count = 0

        # --- Physical constants ---
        self.V_pre = tf.constant(V_pre, dtype=tf.float64)
        self.Vbi = tf.constant(Vbi, dtype=tf.float64)
        self.Vt = tf.constant(Vt, dtype=tf.float64)
        self.n0 = tf.constant(n0, dtype=tf.float64)
        self.nL = tf.constant(nL, dtype=tf.float64)
        self.lambda_ = tf.constant(lambda_, dtype=tf.float64)
        self.I_bulk = tf.constant(I_bulk, dtype=tf.float64)
        self.W_cons = tf.constant(W_cons, dtype=tf.float64)

        self.FI_bc_loss_curve = []
        self.psi_loss_curve = []
        self.n_loss_curve = []
        self.I_loss_curve = []
        self.cons_loss_curve = []
        self.total_loss_curve = []
        self.loss_phase_curve = []   # "adam" or "lbfgs" for each recorded loss
        self.loss_step_curve = []    # optimizer step within that phase

        # Optimizer runtime and termination information (saved to training_metadata.csv)
        self.adam_runtime_s = np.nan
        self.lbfgs_runtime_s = np.nan
        self.total_runtime_s = np.nan
        self.lbfgs_converged = False
        self.lbfgs_failed = False
        self.lbfgs_iterations = 0
        self.lbfgs_objective_evaluations = 0
        self.lbfgs_final_objective = np.nan

        # Pre-compute uniform grid for ion conservation
        self.X_unif_const = tf.cast(tf.concat([
            tf.reshape(tf.linspace(0.0, 1.0, 5000), [-1, 1]),
            tf.zeros((5000, 1))
        ], axis=1), tf.float64)

    def call(self, X):
        return self.ui(X)

    def ui(self, X):
        x = X[:, 0:1]
        t = X[:, 1:2]
        # PreBias is steady-state: t is zeroed out for scaling
        X_scaled = tf.concat([2.0 * x - 1.0, tf.zeros_like(t)], axis=1)

        for l in self.model_layers:
            X_scaled = l(X_scaled)
        return X_scaled

    @tf.function
    def psi_c(self):
        # psi(b) = Vbi + Vi, as in the paper and the FD solver
        return (self.Vbi + self.V_pre) / self.Vt

    def psi(self, X):
        x = X[:, 0:1]
        t = X[:, 1:2]

        psi_left = tf.constant(0.0, dtype=tf.float64)
        psi_right = self.psi_c() * tf.ones_like(x)

        psi_hat = self.ui(X)[:, 0:1]
        d_x = dist(x, 0.0, 1.0)

        # Hard psi BCs
        psi_bc = psi_left + (psi_right - psi_left) * x + d_x * psi_hat

        return psi_bc

    # =========================================================
    # 1. Output Functions (Log-Space)
    # =========================================================
    def qn(self, X):
        x = X[:, 0:1]
        t = X[:, 1:2]

        # Convert physical scaled boundaries to log space
        qn_left = tf.math.log(self.n0)
        qn_right = tf.math.log(self.nL)

        # Network outputs the log variation
        qn_hat = self.ui(X)[:, 1:2]
        d_x = dist(x, 0.0, 1.0)

        # Hard qn BCs (applied purely in log-space)
        qn_bc = qn_left + (qn_right - qn_left) * x + d_x * qn_hat
        return qn_bc

    def qI(self, X):
        # Network outputs the log variation for ions directly.
        # No hard BCs (since we are using the soft FI=0 penalty).
        qI_hat = self.ui(X)[:, 2:3]
        return qI_hat

    @tf.function
    def n(self, X):
        # Exponentiate to recover the true scaled physical value
        return tf.exp(self.qn(X))

    @tf.function
    def I(self, X):
        # Exponentiate to recover the true scaled physical value
        return tf.exp(self.qI(X))

    # =========================================================
    # 2. Log-Space Gradients (Steady-State / 1D spatial)
    # =========================================================
    @tf.function
    def dpsi(self, X):
        with tf.GradientTape() as tape:
            pos = X[:, 0:1]
            t = X[:, 1:2]
            tape.watch(pos)
            Xcat = tf.concat([pos, t], axis=1)
            psi_val = self.psi(Xcat)
        return tape.gradient(psi_val, pos)

    @tf.function
    def d2psi(self, X):
        x = X[:, 0:1]
        t = X[:, 1:2]
        with tf.GradientTape(persistent=True) as g2:
            g2.watch(x)
            with tf.GradientTape() as g1:
                g1.watch(x)
                Xcat = tf.concat([x, t], axis=1)
                psi_val = self.psi(Xcat)
            dpsi_dx = g1.gradient(psi_val, x)
        d2psi_dx2 = g2.gradient(dpsi_dx, x)
        del g2
        return d2psi_dx2

    @tf.function
    def dqn(self, X):
        with tf.GradientTape() as tape:
            pos = X[:, 0:1]
            t = X[:, 1:2]
            tape.watch(pos)
            Xcat = tf.concat([pos, t], axis=1)
            qn_val = self.qn(Xcat)
        return tape.gradient(qn_val, pos)

    @tf.function
    def d2qn(self, X):
        x = X[:, 0:1]
        t = X[:, 1:2]
        with tf.GradientTape(persistent=True) as g2:
            g2.watch(x)
            with tf.GradientTape() as g1:
                g1.watch(x)
                Xcat = tf.concat([x, t], axis=1)
                qn_val = self.qn(Xcat)
            dqn_dx = g1.gradient(qn_val, x)
        d2qn_dx2 = g2.gradient(dqn_dx, x)
        del g2
        return d2qn_dx2

    @tf.function
    def dqI(self, X):
        with tf.GradientTape() as tape:
            pos = X[:, 0:1]
            t = X[:, 1:2]
            tape.watch(pos)
            Xcat = tf.concat([pos, t], axis=1)
            qI_val = self.qI(Xcat)
        return tape.gradient(qI_val, pos)

    @tf.function
    def d2qI(self, X):
        x = X[:, 0:1]
        t = X[:, 1:2]
        with tf.GradientTape(persistent=True) as g2:
            g2.watch(x)
            with tf.GradientTape() as g1:
                g1.watch(x)
                Xcat = tf.concat([x, t], axis=1)
                qI_val = self.qI(Xcat)
            dqI_dx = g1.gradient(qI_val, x)
        d2qI_dx2 = g2.gradient(dqI_dx, x)
        del g2
        return d2qI_dx2

    # =========================================================
    # 3. Flux Calculations
    # =========================================================
    # Log-space boundary penalty for ions (FI = 0)
    @tf.function
    def FI_log(self, X):
        return -(self.dqI(X) + self.dpsi(X))

    # Safe physical flux calculations for plotting/evaluation
    @tf.function
    def jn(self, X):
        n_val = self.n(X)
        return n_val * (self.dqn(X) - self.dpsi(X))

    @tf.function
    def FI(self, X):
        I_val = self.I(X)
        return I_val * self.FI_log(X)

    @tf.function
    def MSE(self, tensor):
        return tf.reduce_mean(tf.math.square(tensor))

    @tf.function
    def get_loss2(self, X_r, X_a, u_a, X_c, u_c, X_0, u_0):
        psi_loss = tf.constant(0.0, dtype=tf.float64)
        n_loss = tf.constant(0.0, dtype=tf.float64)
        I_loss = tf.constant(0.0, dtype=tf.float64)
        FI_bc_loss = tf.constant(0.0, dtype=tf.float64)
        cons_loss = tf.constant(0.0, dtype=tf.float64)

        # 1. Boundary Flux Loss (Log-Space, evaluated using W_FI)
        FI_bc_loss += self.MSE(self.FI_log(X_a)) + self.MSE(self.FI_log(X_c))

        # Extract log-space gradients
        dpsi_dx = self.dpsi(X_r)
        d2psi_dx2 = self.d2psi(X_r)

        dqn_dx = self.dqn(X_r)
        d2qn_dx2 = self.d2qn(X_r)

        dqI_dx = self.dqI(X_r)
        d2qI_dx2 = self.d2qI(X_r)

        # 2. Poisson
        # (self.I and self.n already apply tf.exp() under the hood now)
        poisson_res = (self.lambda_**2) * d2psi_dx2 + (self.I(X_r) - self.n(X_r))
        psi_loss += self.MSE(poisson_res)

        # 3. Steady-State Log Continuity (Electrons)
        n_res = dqn_dx * (dqn_dx - dpsi_dx) + d2qn_dx2 - d2psi_dx2
        n_loss += self.MSE(n_res)

        # 4. Steady-State Log Continuity (Ions)
        I_res = dqI_dx * (dqI_dx + dpsi_dx) + d2qI_dx2 + d2psi_dx2
        I_loss += self.MSE(I_res)

        # 5. Conservation Loss (Your exact trapezoidal implementation)
        I_unif = tf.reshape(self.I(self.X_unif_const), [-1])
        dx = 1.0 / (5000 - 1)
        I_mean_true = tf.reduce_sum(0.5 * (I_unif[1:] + I_unif[:-1])) * dx
        cons_loss += self.W_cons * tf.square(I_mean_true - self.I_bulk)

        # Apply the critical W_FI scaling to the boundary!
        total_loss = psi_loss + n_loss + I_loss + FI_bc_loss + cons_loss

        return total_loss, FI_bc_loss, psi_loss, n_loss, I_loss, cons_loss

    # --------------------------------------------------------
    # Gradient Computations
    # --------------------------------------------------------
    @tf.function
    def get_grad_adam(self, X_r, X_a, u_a, X_c, u_c, X_0, u_0):
        """Used for Adam: Returns gradients PLUS component losses for tracking."""
        with tf.GradientTape() as tape:
            L, FI_loss, psi_loss, n_loss, I_loss, cons_loss = self.get_loss2(X_r, X_a, u_a, X_c, u_c, X_0, u_0)
        g = tape.gradient(L, self.trainable_variables)
        return L, g, FI_loss, psi_loss, n_loss, I_loss, cons_loss

    @tf.function
    def get_grad(self, X_r, X_a, u_a, X_c, u_c, X_0, u_0):
        """Used for L-BFGS: Strictly returns (Loss, Gradients) to satisfy tfp_function_factory."""
        with tf.GradientTape() as tape:
            L, _, _, _, _, _ = self.get_loss2(X_r, X_a, u_a, X_c, u_c, X_0, u_0)
        g = tape.gradient(L, self.trainable_variables)
        return L, g

    # --------------------------------------------------------
    # Training Loops
    # --------------------------------------------------------
    def network_learn(self, X_r, X_a, u_a, X_c, u_c, X_0, u_0):
        for i in range(self.num_epoch_adam):
            L, g, FI_loss, psi_loss, n_loss, I_loss, cons_loss = self.get_grad_adam(X_r, X_a, u_a, X_c, u_c, X_0, u_0)

            self.train_op.apply_gradients(zip(g, self.trainable_variables))

            if i % self.print_epoch == 0:
                self.FI_bc_loss_curve.append(FI_loss.numpy())
                self.psi_loss_curve.append(psi_loss.numpy())  # Fixed: using psi_loss instead of p_loss
                self.n_loss_curve.append(n_loss.numpy())
                self.I_loss_curve.append(I_loss.numpy())
                self.cons_loss_curve.append(cons_loss.numpy())
                self.total_loss_curve.append(L.numpy())
                self.loss_phase_curve.append("adam")
                self.loss_step_curve.append(int(i))

                print(f"Epoch {i} | Total: {L.numpy():.4e} | FI_bc: {FI_loss.numpy():.4e} | Cons: {cons_loss.numpy():.4e} | I: {I_loss.numpy():.4e} | psi: {psi_loss.numpy():.4e} | n: {n_loss.numpy():.4e}")

    def train(self, X_r, X_a, u_a, X_c, u_c, X_0, u_0):
        t0 = time.time()

        print(f"Training (ADAM) for {self.num_epoch_adam} epochs...")
        self.network_learn(X_r, X_a, u_a, X_c, u_c, X_0, u_0)
        t1 = time.time()
        self.adam_runtime_s = float(t1 - t0)
        print(f"ADAM completed in {t1 - t0:.2f} seconds")

        # -------- Optional L-BFGS PHASE -----------
        if self.num_epoch_lbfgs > 0:
            print(f"\nPhase 2: Training (L-BFGS) - Max Iter: {self.num_epoch_lbfgs}")

            loss_func = tfp_function_factory(self, X_r, X_a, u_a, X_c, u_c, X_0, u_0)
            init_params = tf.dynamic_stitch(loss_func.idx, self.trainable_variables)

            results = tfp.optimizer.lbfgs_minimize(
                value_and_gradients_function=loss_func,
                initial_position=init_params,
                max_iterations=self.num_epoch_lbfgs
            )

            loss_func.assign_new_model_parameters(results.position)

            self.lbfgs_converged = bool(results.converged.numpy())
            self.lbfgs_failed = bool(results.failed.numpy())
            self.lbfgs_iterations = int(results.num_iterations.numpy())
            self.lbfgs_objective_evaluations = int(results.num_objective_evaluations.numpy())
            self.lbfgs_final_objective = float(results.objective_value.numpy())

            t2 = time.time()
            self.lbfgs_runtime_s = float(t2 - t1)
            print(f"L-BFGS completed in {t2 - t1:.2f} seconds")
            print(f"L-BFGS status | converged={self.lbfgs_converged} | failed={self.lbfgs_failed} | "
                  f"iterations={self.lbfgs_iterations} | objective evaluations={self.lbfgs_objective_evaluations}")
        else:
            t2 = time.time()
            print("Skipping L-BFGS: Adam-only run.")

        self.total_runtime_s = float(t2 - t0)
        print(f"Total Training Time: {t2 - t0:.2f} seconds")

if __name__ == "__main__":
    #===================================================================
    #               User-defined parameters
    #===================================================================
    data_type = "float64"
    tf.keras.backend.set_floatx(data_type)

    # Layer thickness
    thick_phys = 500e-9
    b = thick_phys

    Vbi = 0.0

    phia = float(globals().get("phia", 0.1))
    phic = float(globals().get("phic", 0.1))

    T = 300
    kB = con.k
    q = con.e
    e0 = con.epsilon_0
    Vt = kB * T / q

    mu_n = 1.0e-4
    Dn = mu_n * Vt
    eps = 50.0 * e0
    DI = 1.0e-16

    # ------------------------------
    # Universal Density Scaling
    # ------------------------------
    C0 = (eps * Vt) / (q * (b**2))
    P0_dim = float(globals().get("P0", 1.0e21))

    # Physical boundary densities
    Nc = 2.1e24
    n0_dim = Nc * np.exp(-phia / Vt)
    nL_dim = Nc * np.exp(-phic / Vt)

    # Find the absolute maximum density in the system
    N_ref = max(C0, P0_dim, n0_dim, nL_dim)

    # Scale ALL densities by N_ref
    n0 = n0_dim / N_ref
    nL = nL_dim / N_ref
    I_bulk = P0_dim / N_ref

    Vi = float(globals().get("Vi", 0.001))
    run_prebias = 0
    train_main_model = False

    V_pre = Vi

    # ------------------------------
    # Transport Scaling
    # ------------------------------

    # The physical Debye Length
    Ld = np.sqrt(eps * Vt / (q * N_ref))

    # The dimensionless Debye length
    lambda_ = Ld / b

    print(f"Ld     = {Ld:.3e} m")
    print(f"lambda_= {lambda_:.3e}")

    # Dimensionless thickness
    thick_scaled = thick_phys / b

    numPts = 100

    xmin = 0.0
    xmax = thick_scaled

    # Make empty lists to store loss values
    FI_intf_loss_curve = []
    psi_int_loss_curve = []
    n_int_loss_curve = []
    I_int_loss_curve = []
    cons_loss_curve = []

    def gen_training_mesh_prebias(thick, numPts):

        x_r = mesh_gen(thick, numPts)
        t0 = tf.zeros_like(x_r, dtype=tf.float64)
        X_r = tf.concat([tf.cast(x_r, tf.float64), t0], axis=1)

        lb = tf.constant([0.0, 0.0], dtype=tf.float64)
        ub = tf.constant([1.0, 0.0], dtype=tf.float64)

        return X_r, lb, ub

    def plot_collocations(X_0, X_r, X_a, X_c):

        plt.scatter(X_r[:, 1], X_r[:, 0], marker='.', alpha=0.3)
        plt.scatter(X_a[:, 1], X_a[:, 0], marker='+')
        plt.scatter(X_c[:, 1], X_c[:, 0], marker='+')
        plt.xlabel('$t$')
        plt.ylabel('$x$')
        plt.title('PreBias collocation points')
        plt.show()

    def calc_psi_c(Va):
        return (Vbi + Va) / Vt

    X_r_pre, lb_pre, ub_pre = gen_training_mesh_prebias(thick_scaled, numPts)
    X_0_pre = X_r_pre

    lb, ub = lb_pre, ub_pre

    X_aPre = tf.constant([[xmin, 0.0]], dtype=tf.float64)
    X_cPre = tf.constant([[xmax, 0.0]], dtype=tf.float64)

    u_aPre = tf.constant([[0.0, n0]], dtype=tf.float64)
    u_cPre = tf.constant([[calc_psi_c(V_pre), nL]], dtype=tf.float64)

    plot_collocations(X_0_pre, X_r_pre, X_aPre, X_cPre)

#Define Pre-Bias Model Class

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

class PreBiasModel:
    def __init__(
        self,
        V_pre, Vbi, Vt, n0, nL, lambda_, N_ref, I_bulk, W_cons,
        hidden_layers=4,
        neurons=20,
        activation="tanh",
        run_lbfgs=True
    ):

        # --------------------------------------------------------
        # Learning Rate Scheduler
        # --------------------------------------------------------
        initial_learning_rate = 1e-3
        decay_steps = 1000
        decay_rate = 0.5

        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate,
            decay_steps=decay_steps,
            decay_rate=decay_rate,
            staircase=True
        )

        self.train_op = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        self.num_epoch_adam = int(globals().get("NUM_EPOCH_ADAM", 2000))
        num_epoch_lbfgs = int(globals().get("NUM_EPOCH_LBFGS", 10000))
        self.num_epoch_lbfgs = num_epoch_lbfgs if run_lbfgs else 0
        self.run_lbfgs = run_lbfgs
        self.print_epoch = 100

        self.Vt = Vt
        self.N_ref = N_ref
        self.I_bulk = I_bulk
        self.W_cons = W_cons

        self.FI_bc_loss_curve = []
        self.psi_loss_curve = []
        self.n_loss_curve = []
        self.I_loss_curve = []
        self.cons_loss_curve = []
        self.total_loss_curve = []
        self.loss_phase_curve = []
        self.loss_step_curve = []

        self.hidden_layers = hidden_layers
        self.neurons = neurons

        activation = activation.lower().strip()
        self.activation = activation

        self.layers = []

        for _ in range(hidden_layers):

            if activation.lower() in ["leaky_relu", "leakyrelu"]:
                self.layers.append(
                    tf.keras.layers.Dense(
                        neurons,
                        activation=None
                    )
                )
                self.layers.append(
                    tf.keras.layers.LeakyReLU(
                        negative_slope=0.01
                    )
                )

            else:
                self.layers.append(
                    tf.keras.layers.Dense(
                        neurons,
                        activation=activation
                    )
                )

        self.layers.append(
            tf.keras.layers.Dense(
                3,
                activation=None
            )
        )

        self.model = model(
            layers=self.layers,
            train_op=self.train_op,
            num_epoch_adam=self.num_epoch_adam,
            num_epoch_lbfgs=self.num_epoch_lbfgs,
            print_epoch=self.print_epoch,
            V_pre=V_pre,
            Vbi=Vbi,
            Vt=Vt,
            n0=n0,
            nL=nL,
            lambda_=lambda_,
            I_bulk=I_bulk,
            W_cons=W_cons
        )

    def train(self, X_r, X_0, X_aPre, u_aPre, X_cPre, u_cPre):
        print("Training PreBias Model...")
        self.model.train(X_r, X_aPre, u_aPre, X_cPre, u_cPre, X_0, None)

        # Copy loss curves from underlying model
        self.FI_bc_loss_curve = self.model.FI_bc_loss_curve.copy()
        self.psi_loss_curve = self.model.psi_loss_curve.copy()
        self.n_loss_curve = self.model.n_loss_curve.copy()
        self.I_loss_curve = self.model.I_loss_curve.copy()
        self.cons_loss_curve = self.model.cons_loss_curve.copy()
        self.total_loss_curve = self.model.total_loss_curve.copy()
        self.loss_phase_curve = self.model.loss_phase_curve.copy()
        self.loss_step_curve = self.model.loss_step_curve.copy()

        # Evaluate steady-state solution on X_0 for export
        psi_0 = self.model.psi(X_0).numpy()
        n_0   = self.model.n(X_0).numpy()
        I_0   = self.model.I(X_0).numpy()

        u_0 = np.concatenate((psi_0, n_0, I_0), axis=1).astype(np.float64)
        x_coords = X_0[:, 0].numpy()

        outdir = globals().get("OUTPUT_DIR", "outputs")
        os.makedirs(outdir, exist_ok=True)

        # -----------------------------
        # Diagnostic PINN-only field plots
        # -----------------------------
        plt.rcParams.update({
            "font.size": 11,
            "axes.labelsize": 11,
            "axes.titlesize": 12,
            "legend.fontsize": 10,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
        })

        fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.0))

        # Convert normalized x to nm
        b_m=globals().get("b", 500e-9)
        x_nm = x_coords * b_m * 1e9

        ax[0].plot(x_nm, psi_0[:, 0] * self.Vt, linewidth=2)
        ax[0].set_title("(a)")
        ax[0].set_xlabel("Position (nm)")
        ax[0].set_ylabel(r"$\psi$ (V)")
        ax[0].grid(True, alpha=0.25)

        ax[1].semilogy(x_nm, n_0[:, 0] * self.N_ref, linewidth=2)
        ax[1].set_title("(b)")
        ax[1].set_xlabel("Position (nm)")
        ax[1].set_ylabel(r"$n$ (m$^{-3}$)")
        ax[1].grid(True, which="both", alpha=0.25)

        ax[2].semilogy(x_nm, I_0[:, 0] * self.N_ref, linewidth=2)
        ax[2].set_title("(c)")
        ax[2].set_xlabel("Position (nm)")
        ax[2].set_ylabel(r"$I$ (m$^{-3}$)")
        ax[2].grid(True, which="both", alpha=0.25)

        plt.tight_layout()
        plt.savefig(
            os.path.join(outdir, "pinn_profiles_diagnostic.png"),
            dpi=400,
            bbox_inches="tight"
        )
        plt.show()

        # -----------------------------
        # Save solution arrays
        # -----------------------------
        np.save(os.path.join(outdir, "u0.npy"), u_0)
        np.save(os.path.join(outdir, "x0.npy"), x_coords.astype(np.float64))

        prebias_weights_path = os.path.join(outdir, "prebias_weights.weights.h5")

        X_dummy = tf.constant([[0.5, 0.0]], dtype=tf.float64)
        _ = self.model(X_dummy)

        self.model.save_weights(prebias_weights_path)

        print(f"Saved PreBias weights to: {prebias_weights_path}")
        print(f"PreBiasModel Completed. u0.npy saved, shape: {u_0.shape}")
        print(f"First row: {u_0[0]}")
        print(f"Last row : {u_0[-1]}")

    def _save_loss_csv(self):
        outdir = globals().get("OUTPUT_DIR", "outputs")
        os.makedirs(outdir, exist_ok=True)

        epochs = np.arange(len(self.total_loss_curve)) * self.model.print_epoch

        loss_df = pd.DataFrame({
            "epoch": epochs,
            "phase": self.loss_phase_curve,
            "step": self.loss_step_curve,
            "total_loss": self.total_loss_curve,
            "L_psi": self.psi_loss_curve,
            "L_n": self.n_loss_curve,
            "L_I": self.I_loss_curve,
            "L_FI": self.FI_bc_loss_curve,
            "L_cons": self.cons_loss_curve,
        })

        loss_df.to_csv(os.path.join(outdir, "loss_history.csv"), index=False)
        print(f"Saved loss history CSV to: {os.path.join(outdir, 'loss_history.csv')}")

        # Optimizer runtimes and L-BFGS stopping information (used by 04_Adam_Epoch_Study)
        metadata_df = pd.DataFrame([{
            "num_epoch_adam": int(self.model.num_epoch_adam),
            "num_epoch_lbfgs_max": int(self.model.num_epoch_lbfgs),
            "adam_runtime_s": float(self.model.adam_runtime_s),
            "lbfgs_runtime_s": float(self.model.lbfgs_runtime_s),
            "total_runtime_s": float(self.model.total_runtime_s),
            "lbfgs_converged": bool(self.model.lbfgs_converged),
            "lbfgs_failed": bool(self.model.lbfgs_failed),
            "lbfgs_iterations": int(self.model.lbfgs_iterations),
            "lbfgs_objective_evaluations": int(self.model.lbfgs_objective_evaluations),
            "lbfgs_final_objective": float(self.model.lbfgs_final_objective),
        }])
        metadata_df.to_csv(os.path.join(outdir, "training_metadata.csv"), index=False)
        print(f"Saved training metadata to: {os.path.join(outdir, 'training_metadata.csv')}")

    def plot_loss_curves_prebias(self, save=True):
        print("Plotting main-paper loss curves for PreBias Model...")

        outdir = globals().get("OUTPUT_DIR", "outputs")
        os.makedirs(outdir, exist_ok=True)

        epochs = np.arange(len(self.total_loss_curve)) * self.model.print_epoch

        plt.rcParams.update({
            "font.size": 11,
            "axes.labelsize": 11,
            "axes.titlesize": 12,
            "legend.fontsize": 9,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
        })

        plt.figure(figsize=(7.0, 4.8))

        plt.plot(epochs, self.total_loss_curve, label="Total loss", linewidth=2.2)
        plt.plot(epochs, self.psi_loss_curve, label=r"$\mathcal{L}_{\psi}$", linewidth=1.8)
        plt.plot(epochs, self.n_loss_curve, label=r"$\mathcal{L}_{n}$", linewidth=1.8)
        plt.plot(epochs, self.I_loss_curve, label=r"$\mathcal{L}_{I}$", linewidth=1.8)

        switch_epoch = self.model.num_epoch_adam
        plt.axvline(x=switch_epoch, linestyle="--", linewidth=1.4, color="gray", label="Adam → L-BFGS")

        plt.yscale("log")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.xlim(left=0)
        plt.grid(True, which="both", alpha=0.25)
        plt.legend(frameon=False)
        plt.tight_layout()

        if save:
            plt.savefig(os.path.join(outdir, "loss_history.png"), dpi=600, bbox_inches="tight")
            plt.savefig(os.path.join(outdir, "loss_history.pdf"), bbox_inches="tight")

        plt.show()

    def plot_loss_curves_prebias_supplementary(self, save=True):
        print("Plotting supplementary loss curves for PreBias Model...")

        outdir = globals().get("OUTPUT_DIR", "outputs")
        os.makedirs(outdir, exist_ok=True)

        epochs = np.arange(len(self.total_loss_curve)) * self.model.print_epoch

        plt.rcParams.update({
            "font.size": 11,
            "axes.labelsize": 11,
            "axes.titlesize": 12,
            "legend.fontsize": 9,
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
        })

        plt.figure(figsize=(7.4, 5.2))

        plt.plot(epochs, self.total_loss_curve, label="Total loss", linewidth=2.2)
        plt.plot(epochs, self.psi_loss_curve, label=r"$\mathcal{L}_{\psi}$", linewidth=1.8)
        plt.plot(epochs, self.n_loss_curve, label=r"$\mathcal{L}_{n}$", linewidth=1.8)
        plt.plot(epochs, self.I_loss_curve, label=r"$\mathcal{L}_{I}$", linewidth=1.8)
        plt.plot(epochs, self.FI_bc_loss_curve, label=r"$\mathcal{L}_{FI}$", linewidth=1.6, linestyle="--")
        plt.plot(epochs, self.cons_loss_curve, label=r"$\mathcal{L}_{\mathrm{cons}}$", linewidth=1.6, linestyle=":")

        switch_epoch = self.model.num_epoch_adam
        plt.axvline(x=switch_epoch, linestyle="--", linewidth=1.4, color="gray", label="Adam → L-BFGS")

        plt.yscale("log")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.xlim(left=0)
        plt.grid(True, which="both", alpha=0.25)
        plt.legend(frameon=False, ncol=2)
        plt.tight_layout()

        if save:
            plt.savefig(
                os.path.join(outdir, "loss_history_supplementary.png"),
                dpi=600,
                bbox_inches="tight"
            )

        plt.show()

#Train Pre-Bias Model

In [ ]:
if __name__ == "__main__":
    if run_prebias == 0:
        print("Running PreBias stage...")

        W_cons = float(globals().get("W_cons", 100.0))
        hidden_layers = int(globals().get("HIDDEN_LAYERS", 4))
        neurons = int(globals().get("NEURONS", 40))
        activation = str(globals().get("ACTIVATION", "tanh"))
        run_lbfgs = bool(globals().get("RUN_LBFGS", True))

        prebias_model = PreBiasModel(
            V_pre, Vbi, Vt, n0, nL, lambda_, N_ref, I_bulk, W_cons,
            hidden_layers=hidden_layers,
            neurons=neurons,
            activation=activation,
            run_lbfgs = run_lbfgs
        )

        prebias_model.train(X_r_pre, X_0_pre, X_aPre, u_aPre, X_cPre, u_cPre)

        I_unif = prebias_model.model.I(prebias_model.model.X_unif_const).numpy().ravel()
        I_mean_pred = np.trapezoid(I_unif, dx=1.0/(len(I_unif)-1))
        print("Predicted mean I (dimensionless):", I_mean_pred)
        print("Target I_bulk:", I_bulk)
        print("Predicted mean I (physical):", I_mean_pred * N_ref, "m^-3")
        print("Target mean I (physical):", I_bulk * N_ref, "m^-3")

        prebias_model._save_loss_csv()
        prebias_model.plot_loss_curves_prebias()
        prebias_model.plot_loss_curves_prebias_supplementary()

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

lb, ub = lb_pre, ub_pre

L = 500
b_m = globals().get('b', 500e-9)

x_nodes_nm    = np.arange(0, L + 1, dtype=float)
x_interior_nm = np.arange(1, L, dtype=float)
x_I_nm        = np.arange(0, L + 1, dtype=float)

Xpsi = tf.convert_to_tensor(
    np.c_[x_nodes_nm * 1e-9 / b_m, np.zeros_like(x_nodes_nm)],
    dtype=tf.float64
)
Xn = tf.convert_to_tensor(
    np.c_[x_interior_nm * 1e-9 / b_m, np.zeros_like(x_interior_nm)],
    dtype=tf.float64
)
XI = tf.convert_to_tensor(
    np.c_[x_I_nm * 1e-9 / b_m, np.zeros_like(x_I_nm)],
    dtype=tf.float64
)

psi_pinn = prebias_model.model.psi(Xpsi).numpy().ravel() * Vt
n_pinn   = prebias_model.model.n(Xn).numpy().ravel() * N_ref
I_pinn   = prebias_model.model.I(XI).numpy().ravel() * N_ref

outdir = globals().get("OUTPUT_DIR", "outputs")
os.makedirs(outdir, exist_ok=True)

pd.DataFrame({'x_nm': x_nodes_nm, 'psi_V': psi_pinn}).to_csv(os.path.join(outdir, "pinn_psi.csv"), index=False)
pd.DataFrame({'x_nm': x_interior_nm, 'n_m3': n_pinn}).to_csv(os.path.join(outdir, "pinn_n.csv"), index=False)
pd.DataFrame({'x_nm': x_I_nm, 'I_m3': I_pinn}).to_csv(os.path.join(outdir, "pinn_I.csv"), index=False)

print("Saved pinn_psi.csv, pinn_n.csv, pinn_I.csv")

In [ ]:
# -------- PINN boundary ionic flux check --------

lb, ub = lb_pre, ub_pre

X_left  = tf.constant([[0.0, 0.0]], dtype=tf.float64)
X_right = tf.constant([[1.0, 0.0]], dtype=tf.float64)

F_left  = prebias_model.model.FI(X_left).numpy().item()
F_right = prebias_model.model.FI(X_right).numpy().item()

print("PINN ionic flux at boundaries (dimensionless):")
print(f"  Left  : {F_left:.4e}")
print(f"  Right : {F_right:.4e}")

I_left  = prebias_model.model.I(X_left).numpy().item()
I_right = prebias_model.model.I(X_right).numpy().item()

print("PINN ion density at boundaries (dimensionless):")
print(f"  Left  : {I_left:.4e}")
print(f"  Right : {I_right:.4e}")